# LangGraph Module 1 — Coding Assessment

## What this assessment is

This is a **practical coding assessment** for Module 1 of the LangGraph: Zero to Hero course. You will write Python code that demonstrates your understanding of the five core LangGraph concepts covered in class: TypedDict state, node functions, graph compilation, conditional routing, and Pydantic structured output.

## Before you start

**Paste the API key provided by your instructor** into the Setup cell below where it says `"sk-proj-paste-your-key-here"`. Do not share this key with anyone. Run the Setup cell first before attempting any questions.

## How to complete this assessment

1. Read each question carefully — the question tells you exactly what to build
2. Write your code in the cell marked `# YOUR CODE HERE`
3. Run your code cell — it will execute immediately
4. The grading cells (marked `# ── Grading cell`) run automatically and show you which checks passed (✅) or failed (❌)
5. Use the feedback from the ❌ items to fix your code and resubmit

## What to submit

When you are satisfied with your answers:

1. Go to **Kernel → Restart & Run All** to execute the entire notebook cleanly from top to bottom
2. Check that no cells produced errors
3. Save the file: **File → Save Notebook**
4. Submit the `.ipynb` file to the **Gradescope assignment** linked from Blackboard

## Scoring

| Question | Topic | Marks |
|----------|-------|-------|
| Q1 | Define a TypedDict state | 4 |
| Q2 | Write a node function | 4 |
| Q3 | Build and compile a graph | 4 |
| Q4 | Write a router function | 4 |
| Q5 | Pydantic structured output | 4 |
| **Total** | | **20** |

Passing mark: **14 / 20 (70%)**. Gradescope will autograde your submission and post the result to Blackboard.

## Rules

- You may refer to your notes and the course notebooks
- Each question must be answered in its own cell — do not restructure the notebook

---


## Setup — run this cell first

In [1]:
# ── Setup — run this cell first ───────────────────────────────────────────────
# Paste the API key your instructor provided below.
# Replace the placeholder text between the quotes with your actual key.

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, Literal, List
from pydantic import BaseModel, Field
import operator

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# # 1. Load local environment variables if they exist
# load_dotenv()

# # 2. Explicitly define the variable so Python never throws a NameError.
# # It checks the system environment, and if it finds nothing, defaults to a safe string.
# OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "dummy_placeholder_key")

# # 3. Instantiate your LLM using the safely defined variable
# llm = ChatOpenAI(api_key=OPENAI_API_KEY, model="gpt-4o-mini", temperature=0)

# ----------------
import os

# # ── Paste your API key here ────────────────────────────────────────────────────
OPENAI_API_KEY = "YOUR_API_KEY_HERE"   # ← replace this with the key from your instructor
# # ──────────────────────────────────────────────────────────────────────────────

if OPENAI_API_KEY == "sk-proj-paste-your-key-here":
    raise ValueError("You have not pasted your API key. Replace the placeholder above with the key provided by your instructor.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
llm = ChatOpenAI(model='gpt-4o-mini')

# ── Grading helper — do not modify ─────────────────────────────────────────────
# Powers the "# ── Grading cell" blocks below. Each check runs in isolation so a
# missing or broken answer is marked ❌ rather than crashing the whole notebook.
_scores = {}

def _check(question, tests, marks_per_test=0.8):
    passed = 0
    print(f'{question} — self-check:')
    for description, test in tests:
        try:
            ok = bool(test())
        except Exception:
            ok = False
        print(f'  {"✅" if ok else "❌"} {description}')
        if ok:
            passed += 1
    score = round(passed * marks_per_test, 2)
    _scores[question] = score
    print(f'  → {passed}/{len(tests)} checks passed  ({score} marks)\n')
    return score

print('✅ Setup complete. You are ready to begin.')


✅ Setup complete. You are ready to begin.


---
## Question 1 — Define a TypedDict state (4 marks)

Define a TypedDict called `BlogState` with these four fields:

- `topic` — a string (the blog topic)
- `word_count` — an integer (target word count)
- `draft` — a string (the generated blog text)
- `approved` — a boolean (whether the draft was approved)

Then create an instance called `initial_state` with:
- `topic = 'Python decorators'`
- `word_count = 300`
- `draft = ''`
- `approved = False`

In [2]:
# YOUR CODE HERE
# Define the state dictionary structure
from typing import TypedDict

# Define BlogState natively as a TypedDict
class BlogState(TypedDict):
    topic: str
    word_count: int
    draft: str
    approved: bool

# Initialize initial_state
initial_state = {
    'topic': 'Python decorators',
    'word_count': 300,
    'draft': '',
    'approved': False
}

In [3]:
# ── Grading cell — do not modify ─────────────────────────────────────────────
_check('Q1', [
    ('BlogState is defined as a TypedDict',
        lambda: 'BlogState' in dir() and issubclass(BlogState, dict)),
    ('BlogState has field: topic (str)',
        lambda: BlogState.__annotations__.get('topic') is str),
    ('BlogState has field: word_count (int)',
        lambda: BlogState.__annotations__.get('word_count') is int),
    ('BlogState has field: approved (bool)',
        lambda: BlogState.__annotations__.get('approved') is bool),
    ('initial_state is created with correct values',
        lambda: initial_state['topic'] == 'Python decorators' and
                initial_state['word_count'] == 300 and
                initial_state['draft'] == '' and
                initial_state['approved'] == False),
], marks_per_test=0.8)

Q1 — self-check:
  ❌ BlogState is defined as a TypedDict
  ✅ BlogState has field: topic (str)
  ✅ BlogState has field: word_count (int)
  ✅ BlogState has field: approved (bool)
  ✅ initial_state is created with correct values


NameError: name '_scores' is not defined

---
## Question 2 — Write a node function (4 marks)

Write a node function called `write_draft` that:

- Takes a `state` dict as its only argument
- Calls `llm.invoke()` with a `HumanMessage` asking it to write a blog post
  about `state['topic']` in approximately `state['word_count']` words
- Returns **only** the delta — a dict with the key `'draft'` containing the LLM's response text

**Important:** nodes must return only the fields they changed, not the full state.

In [ ]:
# YOUR CODE HERE
from langchain_core.messages import HumanMessage

def write_draft(state: dict) -> dict:
    topic = state.get('topic', 'Python decorators')
    word_count = state.get('word_count', 300)
    
    prompt = f"Write a brief blog post draft about {topic} that is approximately {word_count} words long."
    response = llm.invoke([HumanMessage(content=prompt)])
    
    return {"draft": response.content}

In [ ]:
# ── Grading cell — do not modify ─────────────────────────────────────────────
import inspect
_test_state = {'topic': 'sorting algorithms', 'word_count': 50, 'draft': '', 'approved': False}
try:
    _result = write_draft(_test_state)
except Exception as _e:
    _result = {}
    print(f'write_draft raised an error: {_e}')

_check('Q2', [
    ('write_draft is defined as a function',
        lambda: callable(write_draft)),
    ('write_draft accepts one argument (state)',
        lambda: len(inspect.signature(write_draft).parameters) == 1),
    ('write_draft returns a dict',
        lambda: isinstance(_result, dict)),
    ('returned dict contains only the delta key: draft',
        lambda: list(_result.keys()) == ['draft']),
    ('draft value is a non-empty string from the LLM',
        lambda: isinstance(_result.get('draft'), str) and len(_result['draft']) > 20),
], marks_per_test=0.8)

---
## Question 3 — Build and compile a graph (4 marks)

Using the `BlogState` TypedDict and `write_draft` node from Questions 1 and 2,
build a LangGraph workflow called `blog_graph` that:

- Starts at `START`
- Runs `write_draft`
- Ends at `END`

Compile it into a runnable app called `blog_app`.

Then invoke it with `initial_state` from Question 1 and store the result in `result`.
Print the first 200 characters of `result['draft']`.

In [ ]:
# YOUR CODE HERE
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# 1. Instantiate the state blueprint framework
blog_graph = StateGraph(BlogState)

# 2. Register the active node function worker
blog_graph.add_node("write_draft", write_draft)

# 3. Set up path edges
blog_graph.add_edge(START, "write_draft")
blog_graph.add_edge("write_draft", END)

# 4. Compile the runtime architecture application
blog_app = blog_graph.compile()

# 5. Visualizer Step (safely contained to keep the cell robust)
try:
    display(Image(blog_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph compiled cleanly, visual skipping metric caught: {e}")

# 6. CRITICAL: Invoke using your exact original blueprint state mapping
result = blog_app.invoke(initial_state)

# 7. Print the requested snippet slice check
print(result.get('draft', '')[:200])

In [ ]:
# ── Grading cell — do not modify ─────────────────────────────────────────────
from langgraph.graph import StateGraph
_check('Q3', [
    ('blog_graph is a StateGraph',
        lambda: isinstance(blog_graph, StateGraph)),
    ('blog_app is compiled (has invoke method)',
        lambda: hasattr(blog_app, 'invoke')),
    ('result is a dict',
        lambda: isinstance(result, dict)),
    ('result contains a draft key with text',
        lambda: isinstance(result.get('draft'), str) and len(result['draft']) > 50),
    ('result preserves original state fields (topic, word_count)',
        lambda: result.get('topic') == 'Python decorators' and result.get('word_count') == 300),
], marks_per_test=0.8)

---
## Question 4 — Write a router function (4 marks)

A sentiment analyser workflow classifies customer reviews.
The state has a field `sentiment` which will be one of `'positive'`, `'negative'`, or `'neutral'`.

Write a router function called `route_review` that:

- Takes `state` as its argument
- Returns the string `'thank_customer'` if sentiment is `'positive'`
- Returns the string `'escalate_issue'` if sentiment is `'negative'`
- Returns the string `'send_standard_reply'` if sentiment is `'neutral'`

The return type annotation should be `Literal['thank_customer', 'escalate_issue', 'send_standard_reply']`.

In [ ]:
# YOUR CODE HERE
from typing import Literal

def route_review(state) -> Literal['thank_customer', 'escalate_issue', 'send_standard_reply']:
    # 1. Retrieve the sentiment field value from the current state
    sentiment = state.get('sentiment')
    
    # 2. Check conditions and return the target node name string
    if sentiment == 'positive':
        return 'thank_customer'
    elif sentiment == 'negative':
        return 'escalate_issue'
    else:
        return 'send_standard_reply'

In [ ]:
# ── Grading cell — do not modify ─────────────────────────────────────────────
_check('Q4', [
    ('route_review is defined as a function',
        lambda: callable(route_review)),
    ("returns 'thank_customer' for positive sentiment",
        lambda: route_review({'sentiment': 'positive'}) == 'thank_customer'),
    ("returns 'escalate_issue' for negative sentiment",
        lambda: route_review({'sentiment': 'negative'}) == 'escalate_issue'),
    ("returns 'send_standard_reply' for neutral sentiment",
        lambda: route_review({'sentiment': 'neutral'}) == 'send_standard_reply'),
    ('return type annotation is a Literal with 3 options',
        lambda: hasattr(route_review, '__annotations__') and
                'return' in route_review.__annotations__ and
                hasattr(route_review.__annotations__['return'], '__args__') and
                len(route_review.__annotations__['return'].__args__) == 3),
], marks_per_test=0.8)

---
## Question 5 — Pydantic structured output (4 marks)

Define a Pydantic BaseModel called `ReviewAnalysis` with two fields:

- `sentiment` — a `Literal['positive', 'negative', 'neutral']`
- `confidence` — a float between 0 and 1, with a Field description:
  `'Confidence score between 0.0 (uncertain) and 1.0 (certain)'`

Then write a function called `analyse_review` that:

- Takes a `review_text: str` argument
- Uses `llm.with_structured_output(ReviewAnalysis)` to classify the review
- Returns the `ReviewAnalysis` object

Test it by calling `analyse_review('This product is absolutely fantastic!')` and
storing the result in `analysis`. Print the sentiment and confidence.

In [ ]:
# YOUR CODE HERE
from pydantic import BaseModel, Field
from typing import Literal

# 1. Define the validated metadata object class blueprint
class ReviewAnalysis(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral']
    confidence: float = Field(
        description='Confidence score between 0.0 (uncertain) and 1.0 (certain)'
    )

# 2. Structure your model processing node function
def analyse_review(review_text: str) -> ReviewAnalysis:
    # Bind the schema object to guide structured extraction mechanics
    structured_llm = llm.with_structured_output(ReviewAnalysis)
    
    # Execute and return the verified model output directly
    return structured_llm.invoke(review_text)

# 3. Test validation tracking sequence as requested by prompt
analysis = analyse_review('This product is absolutely fantastic!')
print(f"Sentiment: {analysis.sentiment}")
print(f"Confidence: {analysis.confidence}")

In [ ]:
# ── Grading cell — do not modify ─────────────────────────────────────────────
_check('Q5', [
    ('ReviewAnalysis is a Pydantic BaseModel',
        lambda: issubclass(ReviewAnalysis, BaseModel)),
    ('ReviewAnalysis has sentiment field with Literal type',
        lambda: 'sentiment' in ReviewAnalysis.model_fields and
                hasattr(ReviewAnalysis.model_fields['sentiment'].annotation, '__args__')),
    ('ReviewAnalysis has confidence field',
        lambda: 'confidence' in ReviewAnalysis.model_fields),
    ('analyse_review returns a ReviewAnalysis object',
        lambda: isinstance(analysis, ReviewAnalysis)),
    ('sentiment is one of the three valid values',
        lambda: analysis.sentiment in ['positive', 'negative', 'neutral']),
    ('confidence is a float between 0 and 1',
        lambda: isinstance(analysis.confidence, float) and 0.0 <= analysis.confidence <= 1.0),
], marks_per_test=4/6)

---
## Results — run this cell last

In [ ]:
# ── Final results — do not modify ───────────────────────────────────────────
_questions = ['Q1','Q2','Q3','Q4','Q5']
_max = 4
_total = sum(_scores.get(q, 0) for q in _questions)
_possible = len(_questions) * _max
_pct = round((_total / _possible) * 100, 1)

# Enter your name here before running this cell
STUDENT_NAME = "Your Name Here"

print('=' * 52)
print('  MODULE 1 ASSESSMENT — RESULTS SUMMARY')
print('=' * 52)
print(f'  Student:  {STUDENT_NAME}')
print()
for q in _questions:
    score = _scores.get(q, 0)
    bar = '█' * int(score) + '░' * int(_max - score)
    print(f'  {q}  [{bar}]  {round(score,1):>4} / {_max}')
print()
print(f'  TOTAL     {_total:.1f} / {_possible}   ({_pct}%)')
print()
if _pct >= 85:
    print('  DISTINCTION — excellent understanding of all concepts.')
elif _pct >= 70:
    print('  PASS — good understanding. Review any ❌ items above.')
else:
    print('  REFER — revisit the notebooks for questions you did not pass.')
print('=' * 52)
